## Building the Baseline Logistic regression model 

In [15]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import pickle

In [16]:
# Load preprocessed dataset
data_preprocessed = pd.read_csv('/Users/arpitalonakadi/Downloads/employee_preprocessed_data.csv')

# Preview first few rows
data_preprocessed.head()

,Reason_1,Reason_2,Reason_3,Reason_4,Month Value,Day of the Week,Transportation Expense,Distance to Work,Age,Daily Work Load Average,Body Mass Index,Education,Children,Pets,Absenteeism Time in Hours
0,0,0,0,1,7,1,289,36,33,239.554,30,0,2,1,4
1,0,0,0,0,7,1,118,13,50,239.554,31,0,1,0,0
2,0,0,0,1,7,2,179,51,38,239.554,31,0,0,0,2
3,1,0,0,0,7,3,279,5,39,239.554,24,0,2,0,4
4,0,0,0,1,7,3,289,36,33,239.554,30,0,2,1,2


In [17]:
# Step 3: Create target variable based on median absenteeism
# This creates a binary target: 1 if absenteeism is above median, else 0

median_absence = data_preprocessed['Absenteeism Time in Hours'].median()
targets = np.where(data_preprocessed['Absenteeism Time in Hours'] > median_absence, 1, 0)

# Add it to the original DataFrame
data_preprocessed['Excessive Absenteeism'] = targets

# Check balance of classes (how many 1s and 0s)
print("Proportion of 1s (excessive absence):", targets.sum() / targets.shape[0])

Proportion of 1s (excessive absence): 0.45571428571428574


This implies that our target variable does now have a class imbalance .. and we can prioceed with creating the regression mdoel 

In [18]:
# Step 4: Drop the original target column, keep only inputs and new target
data_with_targets = data_preprocessed.drop(['Absenteeism Time in Hours'], axis=1)

# Quick sanity check: this should return False (means new df is independent)
print(data_with_targets is data_preprocessed)

False


In [19]:
#Preparing the features (X) and the target (Y)
# Select all columns except the target column
unscaled_inputs = data_with_targets.iloc[:, :-1]

#### Standardizing

Standardization is a widely used preprocessing technique in machine learning. When features have different scales or magnitudes, models may become biased toward those with larger values. To avoid this, it’s important to bring all inputs to a similar scale. This is especially important because many machine learning algorithms perform poorly on unscaled data.
The StandardScaler from sklearn is a powerful tool for this purpose, offering more flexibility than basic preprocessing methods.

Here we will be creating a custom scalar based on our formula. Standardisation can be done based on problem requirement and custom scalar can thus be adjusted respectively. 

In [24]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

class CustomScaler(BaseEstimator, TransformerMixin): 
    def __init__(self, columns, copy=True, with_mean=True, with_std=True):
        # Initialize the standard scaler with correct keyword arguments
        self.scaler = StandardScaler(copy=copy, with_mean=with_mean, with_std=with_std)
        self.columns = columns
        self.mean_ = None
        self.var_ = None

    def fit(self, X, y=None):
        self.scaler.fit(X[self.columns], y)
        self.mean_ = np.mean(X[self.columns])
        self.var_ = np.var(X[self.columns])
        return self

    def transform(self, X, y=None, copy=None):
        init_col_order = X.columns
        X_scaled = pd.DataFrame(self.scaler.transform(X[self.columns]), columns=self.columns)
        X_not_scaled = X.loc[:, ~X.columns.isin(self.columns)]
        return pd.concat([X_not_scaled, X_scaled], axis=1)[init_col_order]

we fit and transform the features using the custom scalar 

In [26]:
# Define which columns to skip scaling
columns_to_omit = ['Reason_1', 'Reason_2', 'Reason_3', 'Reason_4', 'Education']

# Scale everything else
columns_to_scale = [x for x in unscaled_inputs.columns.values if x not in columns_to_omit]

# Initialize and fit the custom scaler
absenteeism_scaler = CustomScaler(columns_to_scale)
absenteeism_scaler.fit(unscaled_inputs)

# Apply transformation
scaled_inputs = absenteeism_scaler.transform(unscaled_inputs)

/Users/arpitalonakadi/Library/Python/3.9/lib/python/site-packages/numpy/_core/fromnumeric.py:4006: FutureWarning: The behavior of DataFrame.var with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return var(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)


splitting the data into test and train 

In [27]:
x_train, x_test, y_train, y_test = train_test_split(scaled_inputs, targets, 
                                                    test_size=0.2, random_state=20)

#### Baseline model 

In [28]:
reg = LogisticRegression()
reg.fit(x_train, y_train)

LogisticRegression()

In [29]:
print("Training Accuracy:", reg.score(x_train, y_train))

Training Accuracy: 0.775


In [30]:
# Get coefficients and interpret
feature_name = unscaled_inputs.columns.values
summary_table = pd.DataFrame(columns=['Feature name'], data=feature_name)
summary_table['Coefficient'] = np.transpose(reg.coef_)

# Add intercept and sort
summary_table.index = summary_table.index + 1
summary_table.loc[0] = ['Intercept', reg.intercept_[0]]
summary_table = summary_table.sort_index()

# Calculate odds ratio
summary_table['Odds_ratio'] = np.exp(summary_table['Coefficient'])

# Sort by impact
summary_table.sort_values('Odds_ratio', ascending=False)

,Feature name,Coefficient,Odds_ratio
3,Reason_3,3.096739,22.125672
1,Reason_1,2.801363,16.467081
2,Reason_2,0.933541,2.543499
4,Reason_4,0.857183,2.356513
7,Transportation Expense,0.613216,1.846359
13,Children,0.361898,1.436052
11,Body Mass Index,0.271155,1.311478
5,Month Value,0.166403,1.181049
10,Daily Work Load Average,-0.000077,0.999923
8,Distance to Work,-0.007779,0.992251


In [31]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
print(classification_report(y_test, reg.predict(x_test)))
print("ROC AUC:", roc_auc_score(y_test, reg.predict_proba(x_test)[:,1]))

              precision    recall  f1-score   support

           0       0.74      0.78      0.76        74
           1       0.74      0.70      0.72        66

    accuracy                           0.74       140
   macro avg       0.74      0.74      0.74       140
weighted avg       0.74      0.74      0.74       140

ROC AUC: 0.7923832923832924


	•	The top positive predictors of excessive absenteeism (based on high positive coefficients and odds ratios) are:
	•	Reason_3 (Odds Ratio: 22.13)
	•	Reason_1 (16.47)
	•	Reason_2, Reason_4, Transportation Expense, Children
	•	Features with negligible or negative influence (likely not adding value):
	•	Daily Work Load Average (~1.00)
	•	Distance to Work, Age, Day of the Week
	•	Education, Pets
	•	Intercept is negative, meaning the baseline probability of absenteeism is low without any influential features.
We see that we can still eliminate a few redundant features and we will try to improve this model by dropping low impact features, including regularization and incorporating threshold values in the next section 


### Summary of Baseline Logistic Regression Model Building

###  1. Load and Inspect the Data
- Loaded the preprocessed absenteeism dataset from a CSV file.
- Used `.head()` to preview the first few rows of the dataset.

###  2. Create Target Variable
- Calculated the median of the `Absenteeism Time in Hours` column.
- Created a new binary target column `Excessive Absenteeism`:
  - `1` if absenteeism > median (excessive).
  - `0` otherwise (normal).
- This approach ensures a **balanced dataset** for training.

###  3. Prepare Inputs for the Model
- Dropped irrelevant or low-impact columns like:
  - `'Absenteeism Time in Hours'`
  - `'Day of the Week'`
  - `'Distance to Work'`, etc.
- Selected all other columns as **features** (inputs).

###  4. Standardize the Data
- Created a **CustomScaler class** based on sklearn’s `StandardScaler`.
- Scaled only selected numerical features (excluded one-hot encoded and categorical columns).
- Ensured input features are of similar magnitude to improve model performance.

###  5. Train-Test Split
- Split the dataset into training and test sets using `train_test_split`.
  - **80%** training data.
  - **20%** test data.
- Set `random_state=20` for reproducibility.

###  6. Train the Logistic Regression Model
- Initialized and trained a `LogisticRegression` model from sklearn on the scaled training data.

###  7. Evaluate the Model
- Evaluated model using:
  - **Accuracy**
  - **Precision, Recall, F1-score** (via `classification_report`)
  - **ROC AUC Score**
- Extracted and analyzed:
  - Model **coefficients** (weights).
  - Corresponding **odds ratios** to understand feature importance.

---